# Proof: RocmPilot-patched nanoGPT runs on AMD
Clones nanoGPT, applies RocmPilot's device fix (`device='cuda'` -> guarded), and
runs nanoGPT's OWN sampler on the AMD GPU — generating real text. This is the
'it actually runs on AMD, not just a report' moment. Run in an AMD ROCm PyTorch Lab.

Handles sandboxed Labs with no system CA bundle (certifi for Python SSL, git verify off).

In [ ]:
import os, re, subprocess, sys

# 0) Fix SSL in a sandboxed Lab: certifi for Python, disable verify for git.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'certifi', 'numpy', 'transformers', 'tiktoken'], check=True)
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# 1) Clone nanoGPT (public).
if not os.path.isdir('nanoGPT'):
    subprocess.run(['git', '-c', 'http.sslVerify=false', 'clone', '--depth', '1',
                    'https://github.com/karpathy/nanoGPT'], check=True)
os.chdir('nanoGPT')

# 2) Apply RocmPilot's device patch to sample.py (device='cuda' -> guarded lookup).
src = open('sample.py').read()
src = re.sub(r"""device\s*=\s*['\"]cuda['\"]""",
             'device = "cuda" if torch.cuda.is_available() else "cpu"', src, count=1)
open('sample.py', 'w').write(src)
print('patched ->', [l for l in src.splitlines() if 'is_available' in l][:1])

In [ ]:
# 3) Run nanoGPT's own sampler ON THE AMD GPU — downloads GPT-2 weights, generates text.
#    float32 for max reliability; --device=cuda runs on the Radeon via ROCm/HIP.
print('=== nanoGPT generating on AMD (Radeon / ROCm) ===\n')
subprocess.run([sys.executable, 'sample.py',
                '--init_from=gpt2', '--device=cuda',
                '--num_samples=1', '--max_new_tokens=60',
                '--dtype=float32'], check=True)

## ✏️ Generate with your own prompt
Edit `PROMPT` (and the knobs) below and run — GPT-2 is already cached, so this is
instant, and it runs on the AMD GPU. Try bigger models with `MODEL`.

In [ ]:
import os, subprocess, sys

# --- edit me ---
PROMPT      = "AMD GPUs are great for AI because"
MODEL       = "gpt2"        # gpt2 | gpt2-medium | gpt2-large | gpt2-xl
NUM_SAMPLES = 2
MAX_TOKENS  = 150
TEMPERATURE = 0.8
# ---------------

# Make sure we're inside the nanoGPT repo (works whether or not the kernel restarted).
if os.path.basename(os.getcwd()) != 'nanoGPT' and os.path.isdir('nanoGPT'):
    os.chdir('nanoGPT')

print(f'=== nanoGPT ({MODEL}) generating on AMD — prompt: {PROMPT!r} ===\n')
subprocess.run([sys.executable, 'sample.py',
                f'--init_from={MODEL}', '--device=cuda', '--dtype=float32',
                f'--start={PROMPT}',
                f'--num_samples={NUM_SAMPLES}',
                f'--max_new_tokens={MAX_TOKENS}',
                f'--temperature={TEMPERATURE}'], check=True)